In [1]:
# =====================================================================
# CELL 1 — Install Packages + Mount Drive + Cache Everything to Drive
# =====================================================================
!pip install -U "transformers>=4.48.0" accelerate bitsandbytes qwen-vl-utils datasets tqdm -q

import os
from google.colab import drive

drive.mount('/content/drive')

STORAGE_ROOT = "/content/drive/MyDrive/GSV_Math_Model_Cache"
MODEL_CACHE_DIR = f"{STORAGE_ROOT}/model_cache"
RESULTS_ROOT = f"{STORAGE_ROOT}/gsv_math_results"

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = "/content/dataset_cache"  # Dataset goes to temp disk to avoid the 0MB bug

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\nModel cache -> {MODEL_CACHE_DIR}")
print(f"Results     -> {RESULTS_ROOT}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 20.4 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

Model cache -> /content/drive/MyDrive/GSV_Math_Model_Cache/model_cache
Results     -> /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results


In [2]:
# =====================================================================
# CELL 2 — Parsing Logic (Do not change)
# =====================================================================
import re

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    prefixes = ["the answer is", "therefore, the answer is", "so the answer is", "the value is", "answer is", "value is", "equals", "it is"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type, precision=2):
    extraction = str(extraction).strip() if extraction else ""
    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[0] if numbers else cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

In [3]:
# =====================================================================
# CELL 3 — Config (QWEN3-VL-8B)
# =====================================================================
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MAX_NEW_TOKENS = 512

RUN_NAME = "qwen3vl8b_zeroshot_query_fixed"
BATCH_SIZE = 50

RUN_DIR = f"{RESULTS_ROOT}/{RUN_NAME}"
import os
os.makedirs(RUN_DIR, exist_ok=True)

print(f"Target Model: {MODEL_ID}")
print(f"Run name:     {RUN_NAME}")
print(f"Checkpoints:  {RUN_DIR}")

Target Model: Qwen/Qwen3-VL-8B-Instruct
Run name:     qwen3vl8b_zeroshot_query_fixed
Checkpoints:  /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed


In [4]:
# =====================================================================
# CELL 4 — Load Dataset and Model (4-bit) — Qwen3-VL
# =====================================================================
from datasets import load_dataset
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

print("Loading MathVista testmini dataset...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
print(f"Loaded {len(mathvista)} test samples.")

print(f"\nLoading model: {MODEL_ID} in 4-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model and Processor loaded successfully.")

Loading MathVista testmini dataset...


Generating testmini split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

Loaded 1000 test samples.

Loading model: Qwen/Qwen3-VL-8B-Instruct in 4-bit...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 4902.28 MB. The target location /content/drive/MyDrive/GSV_Math_Model_Cache/model_cache/hub/models--Qwen--Qwen3-VL-8B-Instruct/blobs only has 2571.55 MB free disk space.
  warnings.warn(


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Model and Processor loaded successfully.


In [5]:
# =====================================================================
# CELL 5 — Evaluation Loop (With Batch Checkpointing to Drive)
# =====================================================================
import gc, json
from pathlib import Path
from tqdm import tqdm
from qwen_vl_utils import process_vision_info
import torch

def load_completed_pids():
    done = set()
    for f in Path(RUN_DIR).glob("batch_*.json"):
        try:
            with open(f) as fh:
                batch = json.load(fh)
            done.update(item["question_id"] for item in batch)
        except json.JSONDecodeError:
            print(f"  [warn] {f.name} looked incomplete, ignoring and re-running its samples.")
    return done

def next_batch_index():
    existing = list(Path(RUN_DIR).glob("batch_*.json"))
    if not existing:
        return 0
    return max(int(f.stem.split("_")[1]) for f in existing) + 1

completed_pids = load_completed_pids()
print(f"Resuming run '{RUN_NAME}': {len(completed_pids)}/{len(mathvista)} samples already done.")

remaining = [s for s in mathvista if s["pid"] not in completed_pids]
print(f"{len(remaining)} samples left to run.\n")

batch_idx = next_batch_index()
batch_results = []

for i, sample in enumerate(tqdm(remaining, desc="Evaluating")):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": sample["decoded_image"],
                    "max_pixels": 313600
                },
                {"type": "text", "text": sample["query"]}
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    generated_ids = [output_ids[j][len(inputs.input_ids[j]):] for j in range(len(output_ids))]
    raw_answer = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    normalized_pred = normalize_extracted_answer(
        raw_answer, sample.get("choices", []), sample["question_type"], sample["answer_type"]
    )
    correct_flag = is_correct(normalized_pred, sample["answer"], sample["answer_type"])

    batch_results.append({
        "question_id": sample["pid"],
        "skills": sample["metadata"]["skills"],
        "correct": correct_flag,
        "raw_answer": raw_answer
    })

    del inputs, output_ids, generated_ids, messages, text_prompt, image_inputs
    gc.collect()
    torch.cuda.empty_cache()

    # CHECKPOINT: write to Drive every BATCH_SIZE samples
    if len(batch_results) >= BATCH_SIZE or i == len(remaining) - 1:
        out_file = f"{RUN_DIR}/batch_{batch_idx:04d}.json"
        with open(out_file, "w") as fh:
            json.dump(batch_results, fh)
        print(f"  [checkpoint] saved {out_file}  ({len(batch_results)} samples)")
        batch_idx += 1
        batch_results = []

print("\nBatch complete for this session. Re-run this cell any time to continue.")

Resuming run 'qwen3vl8b_zeroshot_query_fixed': 950/1000 samples already done.
50 samples left to run.



Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
Evaluating: 100%|██████████| 50/50 [32:33<00:00, 39.08s/it]

  [checkpoint] saved /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed/batch_0019.json  (50 samples)

Batch complete for this session. Re-run this cell any time to continue.


In [6]:
# =====================================================================
# CELL 6 — Aggregate ALL saved batches from Drive
# =====================================================================
import json, glob

results = []
for f in sorted(glob.glob(f"{RUN_DIR}/batch_*.json")):
    with open(f) as fh:
        results.extend(json.load(fh))

print(f"Aggregated {len(results)} / {len(mathvista)} total samples from {RUN_DIR}")

if len(results) < len(mathvista):
    print("\n  Run is NOT complete yet. Re-run Cell 5 (after re-running 1-4 if you reconnected)")
    print("  to continue -- it will automatically skip everything already saved.")
else:
    print("\n  Run complete -- all 1000 samples present. Saving consolidated final file...")
    final_path = f"{RUN_DIR}/FINAL_results.json"
    with open(final_path, "w") as fh:
        json.dump(results, fh)
    print(f"  Saved: {final_path}")

Aggregated 1000 / 1000 total samples from /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed

  Run complete -- all 1000 samples present. Saving consolidated final file...
  Saved: /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed/FINAL_results.json


In [7]:
# =====================================================================
# CELL 7 — Calculate and Print Metrics
# =====================================================================
skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    correct = res["correct"]
    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1

    for skill in res.get("skills", []):
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*55)
print(f"{MODEL_ID + ' — Zero-Shot MathVista (' + RUN_NAME + ')':^55}")
print("="*55)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-"*55)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*55)
print(f"\nFull results saved at:\n{RUN_DIR}")


Qwen/Qwen3-VL-8B-Instruct — Zero-Shot MathVista (qwen3vl8b_zeroshot_query_fixed)
Category             | Correct    | Total      | Accuracy  
-------------------------------------------------------
All                  | 397        | 1000       | 39.70%
Geometry             | 93         | 239        | 38.91%
Arithmetic           | 125        | 353        | 35.41%
Algebra              | 111        | 281        | 39.50%
Logic                | 7          | 37         | 18.92%
Numeric              | 46         | 144        | 31.94%
Scientific           | 54         | 122        | 44.26%
Statistical          | 127        | 301        | 42.19%

Full results saved at:
/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed


In [8]:
# =====================================================================
# RE-SCORE QWEN3-VL WITH v2 PARSER
# =====================================================================
import json, glob, re
from tqdm import tqdm
from datasets import load_dataset

RUN_DIR = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen3vl8b_zeroshot_query_fixed"

# --- v2 Parser Logic ---
FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches: return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form_v2(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    prefixes = ["the answer is", "therefore, the answer is", "so the answer is",
                "the value is", "answer is", "value is", "equals", "it is",
                "the final answer is", "final answer:", "answer:"]
    for prefix in prefixes:
        if text.startswith(prefix): text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar_v2(extraction, choices):
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_v2(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)
    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar_v2(clean_free_form_v2(extraction), choices)
    else:
        cleaned = clean_free_form_v2(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
    return extraction

def is_correct_v2(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

# --- Execution ---
print("Loading dataset for ground truth answers...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
pid_lookup = {s["pid"]: s for s in mathvista}

all_results = []
for f in sorted(glob.glob(f"{RUN_DIR}/batch_*.json")):
    with open(f) as fh: all_results.extend(json.load(fh))

print(f"\nLoaded {len(all_results)} saved results.")

old_correct = sum(r["correct"] for r in all_results)
new_correct = 0

for r in tqdm(all_results, desc="Re-scoring with v2 parser"):
    sample = pid_lookup.get(r["question_id"])
    if not sample: continue
    new_pred = normalize_v2(r["raw_answer"], sample.get("choices", []), sample["question_type"], sample["answer_type"])
    new_correct += is_correct_v2(new_pred, sample["answer"], sample["answer_type"])

print("\n" + "="*60)
print(f"{'Qwen3-VL-8B — Zero-Shot (v2 Parser)':^60}")
print("="*60)
print(f"Old parser score: {old_correct}/1000 = {old_correct/10:.1f}%")
print(f"New parser score: {new_correct}/1000 = {new_correct/10:.1f}%")
print(f"Recovered:        +{new_correct - old_correct} samples")
print("="*60)

# The moment of truth: which model won?
print("\nFINAL ZERO-SHOT STANDINGS:")
print(f"1. Qwen2.5-VL-7B : 61.9%")
print(f"2. Qwen3-VL-8B   : {new_correct/10:.1f}%")

Loading dataset for ground truth answers...

Loaded 1000 saved results.


Re-scoring with v2 parser: 100%|██████████| 1000/1000 [00:00<00:00, 3212.32it/s]


            Qwen3-VL-8B — Zero-Shot (v2 Parser)             
Old parser score: 397/1000 = 39.7%
New parser score: 588/1000 = 58.8%
Recovered:        +191 samples

FINAL ZERO-SHOT STANDINGS:
1. Qwen2.5-VL-7B : 61.9%
2. Qwen3-VL-8B   : 58.8%
